In [1]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/707.8 kB ? eta -:--:--
   ---------------------------------------- 707.8/707.8 kB 7.0 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 17.4 MB/s  0:00:00

   ---- -----------------------------------  1/10 [wassima]
   -------- -------------------------------  2/10 [qh3]
   -------- -------------------------------  2/10 [qh3]
   ---------------- -----------------------  4/10 [jh2]
   ---------------- -----------------------  4/10 [jh2]
   ------------------------ ---------------  6/10 [charset-normalizer]
   ---------------------------- -----------  7/10 [urllib3-future]
   ---------------------------- -----------  7/10 [urllib3-future]
   ---------------------------- -----------  7/10 [urllib3-future]
   ---------------------------- ----------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached numpy-2.4.3-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.1-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached numpy-2.4.3-cp313-cp313-win_amd64.whl (12.3 MB)
Using cached pandas-3.0.1-cp313-cp313-win_amd64.whl (9.7 MB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
Using cach


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": -6,
	"longitude": 35,
	"hourly": ["temperature_2m", "relative_humidity_2m", "rain", "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm"],
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_rain = hourly.Variables(2).ValuesAsNumpy()
hourly_soil_temperature_0cm = hourly.Variables(3).ValuesAsNumpy()
hourly_soil_temperature_6cm = hourly.Variables(4).ValuesAsNumpy()
hourly_soil_temperature_18cm = hourly.Variables(5).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["rain"] = hourly_rain
hourly_data["soil_temperature_0cm"] = hourly_soil_temperature_0cm
hourly_data["soil_temperature_6cm"] = hourly_soil_temperature_6cm
hourly_data["soil_temperature_18cm"] = hourly_soil_temperature_18cm

hourly_dataframe = pd.DataFrame(data = hourly_data)
hourly_dataframe


Coordinates: -6.0°N 35.0°E
Elevation: 1035.0 m asl
Timezone difference to GMT+0: 0s


,date,temperature_2m,relative_humidity_2m,rain,soil_temperature_0cm,soil_temperature_6cm,soil_temperature_18cm
0,2026-03-16 00:00:00+00:00,18.983000,93.0,0.0,18.632999,20.733000,22.683001
1,2026-03-16 01:00:00+00:00,18.983000,91.0,0.0,18.632999,20.483000,22.483000
2,2026-03-16 02:00:00+00:00,18.782999,91.0,0.0,18.433001,20.333000,22.333000
3,2026-03-16 03:00:00+00:00,18.683001,91.0,0.0,18.333000,20.132999,22.183001
4,2026-03-16 04:00:00+00:00,18.833000,91.0,0.0,18.733000,20.032999,22.032999
...,...,...,...,...,...,...,...
163,2026-03-22 19:00:00+00:00,21.983000,80.0,0.0,20.733000,22.782999,24.032999
164,2026-03-22 20:00:00+00:00,21.782999,82.0,0.0,20.733000,22.583000,23.882999
165,2026-03-22 21:00:00+00:00,21.583000,84.0,0.0,20.733000,22.382999,23.733000
166,2026-03-22 22:00:00+00:00,21.183001,86.0,0.0,20.483000,22.132999,23.583000
